# ValleyFever-NewsCast

## 1. Google News Scraping

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [1]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/ValleyFever-NewsCast/'):
        !git clone https://github.com/Adrian1840/ValleyFever-NewsCast
    os.chdir('/content/ValleyFever-NewsCast')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Cloning into 'ValleyFever-NewsCast'...
remote: Enumerating objects: 533, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 533 (delta 84), reused 1 (delta 1), pack-reused 368 (from 3)
Receiving objects: 100% (533/533), 3.86 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (222/222), done.
Current working directory: /content/ValleyFever-NewsCast


## Import Libraries

In [2]:
!pip install pygooglenews #Google News Scraper
!pip install googlenewsdecoder # Converts Redirect Link to Actual Link
!pip install newspaper4k # Scrapes Full article text

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.8 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=ed6287a0988400fe45353894bb39cabacf1e85531dcd2b8786986a11592a9a4c
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of lxml-html-clean to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
  # Basic Packages
  import numpy as np
  import pandas as pd
  import time, random #for adding small delay to avoid rapid fire searches

  #Google News packages
  from pygooglenews import GoogleNews
  import googlenewsdecoder
  from newspaper import Article

  from tqdm import tqdm #Adds progress bar

### Months of Interest

In [ ]:
#includes 3 months before VF Case Rate data begins (3 months before 2008-10) for time-lags
months = pd.period_range("2008-07", "2015-12", freq="M").astype(str)

## Google News Article Info Scraping using `pygooglenews`

In [ ]:
gn = GoogleNews(lang="en", country="US")

In [ ]:
def month_articles_and_count(ym):
    """
    Collect article-level Google News results for one month and return both:
    1. article-level dataframe
    2. monthly article count
    """

    start_date = ym + "-01"
    end_date = (pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")

    search_terms = [
        '"valley fever"',
        "coccidioidomycosis",
        '"San Joaquin Valley" fever'
    ]

    rows = []

    for term in search_terms:
        try:
            res = gn.search(term, from_=start_date, to_=end_date)
            entries = res.get("entries", [])

            for e in entries:
                rows.append({
                    "Year-Month": ym,
                    "query_term": term,
                    "title": e.get("title", ""),
                    "published": e.get("published", ""),
                    "link": e.get("link", ""),
                    "source": e.get("source", {}).get("title", "")
                        if isinstance(e.get("source"), dict) else "",
                    "summary": e.get("summary", "")
                })

            time.sleep(random.uniform(1.5, 3.5)) # adds cool down to avoid getting locked out

        except Exception as ex:
            print(f"Error for {ym}, term={term}: {ex}")

    if not rows:
        article_df = pd.DataFrame(columns=[
            "Year-Month",
            "query_term",
            "title",
            "published",
            "link",
            "source",
            "summary"
        ])
        article_count = 0
        return article_df, article_count

    article_df = pd.DataFrame(rows)

    # Deduplicate repeated articles across search terms !
    if "link" in article_df.columns:
        article_df = article_df.drop_duplicates(subset=["link"])

    article_count = len(article_df)

    return article_df, article_count

## Collecting Monthly Article Count & Article Info

In [ ]:
all_article_dfs = [] #
monthly_counts = [] #article count for a given Year-Month

for ym in months:
    print("Collecting:", ym)

    df_month, count_month = month_articles_and_count(ym) #Returns dataframe of article for that given month, along with num articles within that y-m

    all_article_dfs.append(df_month)

    monthly_counts.append({
        "Year-Month": ym,
        "Num_Articles": count_month
    })

Collecting: 2008-07
Collecting: 2008-08


### Saving Article Info and Article Count File

In [ ]:
# Article-level raw RSS dataset
google_news_rss_raw = pd.concat(all_article_dfs, ignore_index=True)
google_news_rss_raw.to_csv("data/raw/google_news_rss_raw.csv", index=False)

# Monthly article-count dataset
article_counts_only = pd.DataFrame(monthly_counts)
article_counts_only.to_csv("data/raw/article_counts_only.csv", index=False)

google_news_rss_raw.head()

,Year-Month,query_term,title,published,link,source,summary
0,2008-08,"""valley fever""",Anthrax investigation generates valuable foren...,"Wed, 20 Aug 2008 07:00:00 GMT",https://news.google.com/rss/articles/CBMioAFBV...,The NAU Review,"<a href=""https://news.google.com/rss/articles/..."


In [ ]:
article_counts_only.head()

,Year-Month,Num_Articles
0,2008-07,0
1,2008-08,1
2,2008-09,1
3,2008-10,0


### Converting Google Redirect Links to Actual Links

Since we only have **redirect links** in the *link* column and the NLP **newspaper4k** package takes only direct links, we must convert these to direct links.

In [ ]:
def get_real_news_url(google_rss_url):
    try:
        # This specifically handles those long encoded RSS strings
        decoded_data = googlenewsdecoder.new_decoderv1(google_rss_url)

        if decoded_data.get('status') == True:
            return decoded_data.get('decoded_url')
        else:
            return google_rss_url
    except Exception as e:
        print(f"Decoding failed: {e}")
        return google_rss_url

In [ ]:
google_news_rss_raw["real_url"] = google_news_rss_raw["link"].apply(get_real_news_url)

### Redirect Link Versus Real Link

In [ ]:
google_news_rss_raw[["link", "real_url"]]

,link,real_url
0,https://news.google.com/rss/articles/CBMioAFBV...,https://news.nau.edu/anthrax-investigation-gen...
1,https://news.google.com/rss/articles/CBMiZEFVX...,https://www.nytimes.com/2008/09/02/science/02g...


## Scraping Full Article Text using `newspaper4k`

In [ ]:
def get_article_text(url):
    """Function puts FULL (lowercase) article text in its own row for that article"""
    try:
        art = Article(url)
        art.download()
        art.parse()

        time.sleep(random.uniform(0.5, 1.5))  # cool down tori prevent being locked out

        return art.text.lower() #lowercases it (thats it)
    except:
        return ""

In [ ]:
tqdm.pandas() #progress bar

google_news_rss_raw["article_text"] = google_news_rss_raw["real_url"].progress_apply(get_article_text)

100%|██████████| 2/2 [00:04<00:00,  2.19s/it]


## Saving Google News File with Article Text

Let's save a subset of the important article features we will need for cleaning.

In [ ]:
google_news_with_text = google_news_rss_raw[["Year-Month",
            "query_term",
            "title",
             "source",
             "real_url",
             "article_text"]]

google_news_with_text.to_csv("data/raw/google_news_with_text.csv", index=False)

In [ ]:
google_news_with_text.head()

,Year-Month,query_term,title,source,real_url,article_text
0,2008-08,"""valley fever""",Anthrax investigation generates valuable foren...,The NAU Review,https://news.nau.edu/anthrax-investigation-gen...,northern arizona university’s paul keim says t...
1,2008-09,"""valley fever""",Google’s Philanthropy Arm Leads Effort to Use ...,The New York Times,https://www.nytimes.com/2008/09/02/science/02g...,
